In [1]:
import hoda
import tensorly as tl
import mne
%load_ext autoreload
%autoreload 2

!pip install line_profiler
%load_ext line_profiler


#tl.tenalg.set_backend('einsum')
#tl.plugins.use_opt_einsum()
    
mne.set_log_level('ERROR')
print(tl.get_backend())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

[notice] A new release of pip is available: 23.2.1 -> 24.2
[notice] To update, run: python3 -m pip install --upgrade pip
The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler
cupy


In [2]:
from moabb.paradigms import P300, LeftRightImagery
from moabb.datasets import *
from mne.decoding import Scaler


paradigm = P300(resample=48)
dataset = BNCI2014009()
#sfreq=500
#paradigm=LeftRightImagery(resample=sfreq)
#dataset = BNCI2014_004()
epochs, labels, meta = paradigm.get_data(
    dataset=dataset, 
     subjects=[1],
     return_epochs=True
)
session = meta['session'][0]
idc = meta['session'] == session
epochs = epochs[idc]
labels = labels[idc]
meta = meta[idc]


BNCI2014009 has been renamed to BNCI2014_009. BNCI2014009 will be removed in version 1.1.
The dataset class name 'BNCI2014009' must be an abbreviation of its code 'BNCI2014-009'. See moabb.datasets.base.is_abbrev for more information.


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


In [3]:
meta

,subject,session,run
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0
...,...,...,...
571,1,0,0
572,1,0,0
573,1,0,0
574,1,0,0


In [4]:
import tensorly.decomposition
import matplotlib.pyplot as plt
import tensorly as tl
import numpy as np
from sklearn.model_selection import train_test_split
from hoda.tensorize import stf_tensor

X = epochs.get_data()
#X = stf_tensor(X, sfreq=sfreq, zscore=False, morlet_params=dict(n_jobs=-1))
X = tl.tensor(X)
y = labels


print(X.shape)
print(X.dtype)

(576, 16, 39)
float32


In [5]:
from sklearn.model_selection import StratifiedKFold
from hoda.hoda import BTTDA, GreedyBTTDA, HODA, trunc_eigh
from sklearn.pipeline import Pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectFwe
from sklearn.preprocessing import StandardScaler

bttda = GreedyBTTDA(
    max_blocks=4,
    truncate=False,
    #rank_grid=[1,2,3],
    rank_grid=None,
    hoda_params=dict(
        rank=None,
        max_iter=256,
        tol=1e-8,
        init ='eye',
        shrinkage='lw',
        toeplitz=None,
        obj='tr',
        solver='lanczos',
        taper=False,
        extra_train_info=False,
        verbose=True,
        random_state=42,
        delta=None,       
    ),
    verbose=True,
    extra_train_info=False,
    cv=StratifiedKFold(random_state=42, shuffle=True),
    n_jobs=1,    
    select=True
)
bttda.fit(X,y)

Model selection block 1/4...


  0%|          | 0/256 [00:00<?, ?it/s]

  0%|          | 0/256 [00:00<?, ?it/s]

  0%|          | 0/256 [00:00<?, ?it/s]

  0%|          | 0/256 [00:00<?, ?it/s]

...

  0%|          | 0/256 [00:00<?, ?it/s]

Fitting block 4/4...


  0%|          | 0/256 [00:00<?, ?it/s]

  0%|          | 0/256 [00:00<?, ?it/s]

GreedyBTTDA(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
            hoda_params={'delta': None, 'extra_train_info': False,
                         'init': 'eye', 'max_iter': 256, 'obj': 'tr',
                         'random_state': 42, 'rank': (2, 2), 'shrinkage': 'lw',
                         'solver': 'lanczos', 'taper': False, 'toeplitz': None,
                         'tol': 1e-08, 'verbose': True},
            max_blocks=4, n_jobs=1, truncate=False, verbose=True)

In [6]:
bttda.model_select_info_

fold  n_features_orig  n_features  \
block rank                                          
0     (1, 1)       0                1           1   
      (2, 2)       0                4           1   
      (4, 4)       0               16           3   
      (8, 8)       0               64           5   
      (16, 16)     0              256          26   
...              ...              ...         ...   
3     (1, 1)       4               73          10   
      (2, 2)       4               76          10   
      (4, 4)       4               88          13   
      (8, 8)       4              136          16   
      (16, 16)     4              328          36   

                                                             hoda  \
block rank                                                          
0     (1, 1)    HODA(obj='tr', random_state=42, rank=(1, 1), v...   
      (2, 2)    HODA(obj='tr', random_state=42, rank=(2, 2), v...   
      (4, 4)    HODA(obj='tr', random_state=42, rank=(4, 4), v...   
      (8, 8)    HODA(obj='tr', random_state=42, rank=(8, 8), v...   
      (16, 16)  HODA(obj='tr', random_state=42, rank=(16, 16),...   
...                                                           ...   
3     (1, 1)    HODA(obj='tr', random_state=42, rank=(1, 1), v...   
      (2, 2)    HODA(obj='tr', random_state=42, rank=(2, 2), v...   
      (4, 4)    HODA(obj='tr', random_state=42, rank=(4, 4), v...   
      (8, 8)    HODA(obj='tr', random_state=42, rank=(8, 8), v...   
      (16, 16)  HODA(obj='tr', random_state=42, rank=(16, 16),...   

                train_score  val_score  
block rank                              
0     (1, 1)       0.961486   0.961458  
      (2, 2)       0.957956   0.964583  
      (4, 4)       0.951446   0.965104  
      (8, 8)       0.909677   0.944792  
      (16, 16)     0.952508   0.931250  
...                     ...        ...  
3     (1, 1)       0.990361   0.976974  
      (2, 2)       0.990733   0.979167  
      (4, 4)       0.990530   0.978070  
      (8, 8)       0.990632   0.979715  
      (16, 16)     0.993709   0.973684  

[100 rows x 6 columns]

In [7]:
df_agg = bttda.model_select_info_.groupby(['block', 'rank', 'n_features'])
df_agg = df_agg.aggregate('mean')
idc = df_agg.groupby('block').val_score.idxmax()
df_select = bttda.model_select_info_.loc[idc]
df_select.groupby(['block', 'rank']).aggregate(['mean', 'std'])

/tmp/ipykernel_4478/4023180376.py:2: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_agg = df_agg.aggregate('mean')
/tmp/ipykernel_4478/4023180376.py:5: FutureWarning: ['hoda'] did not aggregate successfully. If any error is raised this will raise in a future version of pandas. Drop these columns/ops to avoid this warning.
  df_select.groupby(['block', 'rank']).aggregate(['mean', 'std'])


fold           n_features_orig      n_features            \
               mean       std            mean  std       mean       std   
block rank                                                                
0     (16, 16)  2.0  1.581139           256.0  0.0       26.8  3.271085   
1     (8, 8)    2.0  1.581139            68.0  0.0        8.4  1.140175   
2     (16, 16)  2.0  1.581139           324.0  0.0       35.2  4.086563   
3     (8, 8)    2.0  1.581139           136.0  0.0       16.4  2.966479   

               train_score           val_score            
                      mean       std      mean       std  
block rank                                                
0     (16, 16)    0.956336  0.005098  0.898092  0.045984  
1     (8, 8)      0.989585  0.002032  0.967982  0.012367  
2     (16, 16)    0.991902  0.002997  0.965110  0.019804  
3     (8, 8)      0.991731  0.002258  0.966458  0.015529

In [8]:
 		fold 	n_features_orig 	n_features 	train_score 	val_score
		mean 	std 	mean 	std 	mean 	std 	mean 	std 	mean 	std
block 	rank 										
0 	(16, 16) 	2.0 	1.581139 	256.0 	0.0 	26.8 	3.271085 	0.956336 	0.005098 	0.898092 	0.045984
1 	(16, 16) 	2.0 	1.581139 	260.0 	0.0 	25.8 	2.774887 	0.982203 	0.003373 	0.937133 	0.024079
2 	(16, 16) 	2.0 	1.581139 	324.0 	0.0 	36.0 	2.549510 	0.989041 	0.003219 	0.951316 	0.019331
3 	(16, 16) 	2.0 	1.581139 	340.0 	0.0 	38.6 	3.361547 	0.990715 	0.003640 	0.957648 	0.021131

SyntaxError: invalid syntax (3329548680.py, line 1)

In [ ]:
import seaborn as sns 
sns.lineplot(data=df_select, x='block',y='train_score')
sns.lineplot(data=df_select, x='block',y='val_score')

In [ ]:
plt.style.use('default')
sns.relplot(data=bttda.model_select_info_, x='n_features', y='val_score', hue='rank', palette='tab10', col='block', kind='line', col_wrap=3)

In [ ]:
import pandas as pd

df = pd.DataFrame(bttda.train_info_)
df

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('default')
import pandas as pd
df = pd.DataFrame(bttda.train_info_)
df
if bttda.extra_train_info:
    n_samples = X.shape[0]
    sns.lineplot(data=df, x='block', y='nmse')
    plt.show()
    sns.lineplot(data=df,x='block',y='F_rt')

In [ ]:
Xt = bttda.transform(X)
Xt_flat = tl.to_numpy(Xt)

In [ ]:
import scipy.stats
import seaborn as sns
import math
import numpy as np

samples = []
for c in bttda.blocks_[0].classes_:
    samples.append(tl.to_numpy(Xt)[y==c])
F,p = scipy.stats.f_oneway(*samples, axis=0)
F = F.flatten()
p = p.flatten()
p = np.nan_to_num(p, nan=1)
fig, ax = plt.subplots(1,1)
sig_idc = p < 0.05
plt.bar(np.arange(len(F))[sig_idc],F[sig_idc], color='blue')
plt.bar(np.arange(len(F))[~sig_idc],F[~sig_idc], color='red')
plt.yscale('log')


In [ ]:
sns.heatmap(np.corrcoef(Xt_flat, rowvar=False),  cmap='vlag', center=0)

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns
from matplotlib import pyplot as plt

if Xt.shape[-1] > 1:
    n_components = 2
    decomp  = PCA(n_components=n_components, whiten=True)
    Xt_viz = decomp.fit_transform(Xt_flat)
    df = pd.DataFrame(Xt_viz, columns=['PC1', 'PC2'])
    df['label'] = y
    df=df.reset_index()
    print(df)
    sns.scatterplot(data=df, x='PC1', y='PC2', hue='label',)
    sns.kdeplot(data=df, x='PC1', y='PC2', hue='label',alpha=.5)
    ax = plt.gca()    
    ax.spines['bottom'].set_position('zero')
    ax.spines['left'].set_position('zero')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)